# 06 GHSL Multi-City Parameterized Pipeline

**Purpose:** Convert the validated Bengaluru Spark/Sedona workflow into one reusable pipeline for the remaining metropolitan regions.

## Cities and order

1. Hyderabad
2. Pune metropolitan region
3. Chennai metropolitan region
4. Mumbai metropolitan region
5. Delhi NCR
6. Ahmedabad
7. Surat

Jaipur is configured separately as a held-out generalization city. Bengaluru remains the frozen reference implementation.

## Safety contract

- Do not rerun or modify validated Bengaluru outputs.
- Do not launch a city batch until its extent and raw-input inventory pass.
- Process `1975`, `2000`, and `2020` first as a three-epoch proof.
- Run the remaining epochs only after the proof passes.
- Validate existing HDFS output before skipping it.
- Stop at the first unexplained failure.
- Use Spark, Sedona, HDFS, and Parquet. Do not replace raster stages with local Pandas/Rasterio processing.

In [1]:
# CELL 1: COLAB BOOTSTRAP AND REMOTE EXECUTION HELPER
from google.colab import drive
from pathlib import Path
import os
import shutil
import subprocess

drive.mount('/content/drive', force_remount=False)

DRIVE_ROOT = Path('/content/drive/MyDrive')
REMOTE_EXEC = DRIVE_ROOT / 'remote_exec.sh'
DRIVE_KEY = DRIVE_ROOT / 'hadoop-hdfs.pem'
LOCAL_KEY = Path('/content/hadoop-hdfs.pem')

if not REMOTE_EXEC.is_file():
    raise FileNotFoundError(f'Remote helper not found: {REMOTE_EXEC}')
os.chmod(REMOTE_EXEC, 0o700)

if not LOCAL_KEY.is_file():
    if not DRIVE_KEY.is_file():
        raise FileNotFoundError(f'SSH key not found: {DRIVE_KEY}')
    shutil.copy2(DRIVE_KEY, LOCAL_KEY)
os.chmod(LOCAL_KEY, 0o600)

if oct(LOCAL_KEY.stat().st_mode & 0o777) != '0o600':
    raise RuntimeError('SSH key permissions must be 0o600')

def run_remote(command, timeout=300, show_output=True):
    result = subprocess.run(
        ['bash', str(REMOTE_EXEC), command],
        text=True,
        capture_output=True,
        timeout=timeout,
    )
    if show_output and result.stdout:
        print(result.stdout)
    if show_output and result.stderr:
        print('REMOTE STDERR:')
        print(result.stderr)
    if result.returncode != 0:
        raise RuntimeError(
            f'Remote command failed with return code {result.returncode}.'
        )
    return result

print('Google Drive: MOUNTED')
print('Remote helper:', REMOTE_EXEC)
print('SSH key:', LOCAL_KEY)
print('CELL 1: SUCCESS')


Mounted at /content/drive
Google Drive: MOUNTED
Remote helper: /content/drive/MyDrive/remote_exec.sh
SSH key: /content/hadoop-hdfs.pem
CELL 1: SUCCESS


In [2]:
# CELL 2: PROJECT-WIDE CONSTANTS AND CITY MANIFEST
# Extents are intentionally not guessed. They must be populated by the
# standardized extent-discovery stage and then frozen before processing.

HISTORICAL_EPOCHS = [1975, 1980, 1985, 1990, 1995, 2000, 2005, 2010, 2015, 2020]
PROOF_EPOCHS = [1975, 2000, 2020]
REMAINING_EPOCHS = [1980, 1985, 1990, 1995, 2005, 2010, 2015]
PRODUCTS = ['population', 'smod', 'built_surface']

HDFS_ROOT = 'hdfs://localhost:9000'
HDFS_CATALOG = f'{HDFS_ROOT}/data/ghsl/metadata/india/geotiff_catalog'
HDFS_CITY_ROOT = f'{HDFS_ROOT}/data/ghsl/curated/cities'
REMOTE_WORK_ROOT = '/home/ubuntu/ghsl_multicity'

DEFAULT_SPARK = {
    'population_driver_memory': '5g',
    'smod_driver_memory': '3g',
    'built_driver_memory': '3g',
    'enrichment_driver_memory': '2g',
    'analytics_driver_memory': '3g',
    'shuffle_partitions': 4,
    'default_parallelism': 2,
}

CITY_CONFIGS = {
    'hyderabad': {
        'city_name': 'Hyderabad Metropolitan Region',
        'model_role': 'training',
        'processing_order': 1,
        'seed_lon': 78.4867,
        'seed_lat': 17.3850,
    },
    'pune_metro': {
        'city_name': 'Pune Metropolitan Region',
        'model_role': 'training',
        'processing_order': 2,
        'seed_lon': 73.8567,
        'seed_lat': 18.5204,
    },
    'chennai_metro': {
        'city_name': 'Chennai Metropolitan Region',
        'model_role': 'training',
        'processing_order': 3,
        'seed_lon': 80.2707,
        'seed_lat': 13.0827,
    },
    'mumbai_metro': {
        'city_name': 'Mumbai Metropolitan Region',
        'model_role': 'training',
        'processing_order': 4,
        'seed_lon': 72.8777,
        'seed_lat': 19.0760,
    },
    'delhi_ncr': {
        'city_name': 'Delhi NCR',
        'model_role': 'training',
        'processing_order': 5,
        'seed_lon': 77.2090,
        'seed_lat': 28.6139,
    },
    'ahmedabad': {
        'city_name': 'Ahmedabad Metropolitan Region',
        'model_role': 'training',
        'processing_order': 6,
        'seed_lon': 72.5714,
        'seed_lat': 23.0225,
    },
    'surat': {
        'city_name': 'Surat Metropolitan Region',
        'model_role': 'training',
        'processing_order': 7,
        'seed_lon': 72.8311,
        'seed_lat': 21.1702,
    },
    'jaipur': {
        'city_name': 'Jaipur Metropolitan Region',
        'model_role': 'held_out',
        'processing_order': 8,
        'seed_lon': 75.7873,
        'seed_lat': 26.9124,
    },
}

for slug, config in CITY_CONFIGS.items():
    config.update({
        'city_slug': slug,
        'extent_method': '2020_urban_footprint_plus_30km_buffer_snapped_to_1km',
        'buffer_m': 30000,
        'target_crs': 'ESRI:54009',
        'grid_resolution_m': 1000,
        'extent_mollweide': None,
        'expected_grid_cells': None,
        'hdfs_output_base': f'{HDFS_CITY_ROOT}/city={slug}',
        'remote_work_dir': f'{REMOTE_WORK_ROOT}/{slug}',
        'spark': dict(DEFAULT_SPARK),
    })

ACTIVE_CITY_SLUG = 'hyderabad'
ACTIVE_CITY = CITY_CONFIGS[ACTIVE_CITY_SLUG]

print('Active city:', ACTIVE_CITY['city_name'])
print('Role:', ACTIVE_CITY['model_role'])
print('Output base:', ACTIVE_CITY['hdfs_output_base'])
print('Extent frozen:', ACTIVE_CITY['extent_mollweide'] is not None)
print('CELL 2: SUCCESS')


Active city: Hyderabad Metropolitan Region
Role: training
Output base: hdfs://localhost:9000/data/ghsl/curated/cities/city=hyderabad
Extent frozen: False
CELL 2: SUCCESS


In [3]:
# CELL 3: VALIDATE CITY CONFIGURATION WITHOUT PROCESSING
required_keys = {
    'city_name', 'city_slug', 'model_role', 'processing_order',
    'seed_lon', 'seed_lat', 'extent_method', 'buffer_m',
    'target_crs', 'grid_resolution_m', 'extent_mollweide',
    'expected_grid_cells', 'hdfs_output_base', 'remote_work_dir', 'spark'
}

missing = sorted(required_keys - set(ACTIVE_CITY))
if missing:
    raise RuntimeError(f'Missing configuration keys: {missing}')

if ACTIVE_CITY['model_role'] not in {'training', 'held_out'}:
    raise RuntimeError('model_role must be training or held_out')
if ACTIVE_CITY['grid_resolution_m'] != 1000:
    raise RuntimeError('This pipeline expects a 1 km canonical grid')
if ACTIVE_CITY['buffer_m'] <= 0:
    raise RuntimeError('buffer_m must be positive')

print('Configuration structure: PASS')
print('Historical epochs:', HISTORICAL_EPOCHS)
print('Proof epochs:', PROOF_EPOCHS)
print('Remaining epochs:', REMAINING_EPOCHS)
print('No Spark processing executed.')
print('CELL 3: SUCCESS')


Configuration structure: PASS
Historical epochs: [1975, 1980, 1985, 1990, 1995, 2000, 2005, 2010, 2015, 2020]
Proof epochs: [1975, 2000, 2020]
Remaining epochs: [1980, 1985, 1990, 1995, 2005, 2010, 2015]
No Spark processing executed.
CELL 3: SUCCESS


In [4]:
# CELL 4: READ-ONLY INFRASTRUCTURE AND RAW-DATA PREFLIGHT
preflight_command = r"""
set -e
source /home/ubuntu/ghsl-sedona-env.sh
export HADOOP_HOME=/usr/local/hadoop
export PATH=${HADOOP_HOME}/bin:${HADOOP_HOME}/sbin:${PATH}

for daemon in NameNode DataNode SecondaryNameNode; do
    jps | awk '{print $2}' | grep -qx "${daemon}" || {
        echo "ERROR: ${daemon} is not running"
        exit 20
    }
done

hdfs dfsadmin -safemode get | grep -qi 'Safe mode is OFF' || {
    echo 'ERROR: HDFS safe mode is enabled'
    exit 21
}

hdfs dfs -test -e /data/ghsl/metadata/india/geotiff_catalog || {
    echo 'ERROR: GHSL metadata catalog is missing'
    exit 22
}

RAW_FILE_COUNT=$(hdfs dfs -find /data/ghsl/raw/india | grep -Ei '\.(tif|tiff)$' | wc -l | tr -d ' ')
echo "RAW_FILE_COUNT=${RAW_FILE_COUNT}"
echo
free -h
echo
df -h / /tmp
echo
swapon --show || true
echo 'PREFLIGHT: SUCCESS'
"""

result = run_remote(preflight_command, timeout=300, show_output=True)
print('Return code:', result.returncode)
print('CELL 4: SUCCESS')


RAW_FILE_COUNT=325

               total        used        free      shared  buff/cache   available
Mem:           7.6Gi       1.4Gi       5.5Gi       2.7Mi       940Mi       6.2Gi
Swap:          8.0Gi       593Mi       7.4Gi

Filesystem      Size  Used Avail Use% Mounted on
/dev/root        77G   24G   54G  31% /
tmpfs           3.8G  656K  3.8G   1% /tmp

NAME      TYPE SIZE   USED PRIO
/swapfile file   8G 593.7M   -1
PREFLIGHT: SUCCESS

REMOTE STDERR:

Return code: 0
CELL 4: SUCCESS


## Extent gate

The original Bengaluru notebooks used a manually validated Mollweide envelope and a fixed expected cell count. Those values cannot be copied to other cities. The next cell deliberately blocks batch generation until the active city's standardized metropolitan extent is discovered, reviewed, and frozen.

The standard rule is:

1. Identify the connected 2020 SMOD urban-centre footprint containing the city seed.
2. Dissolve that component.
3. Buffer by 30 km in Mollweide.
4. Take the envelope and snap its bounds outward to the global 1 km grid.
5. Freeze that single extent for all 1975–2020 epochs.

In [5]:
# CELL 5: EXTENT GATE
# Populate these only from the extent-discovery result, never from a guess.

def assert_extent_frozen(city_config):
    extent = city_config['extent_mollweide']
    expected_cells = city_config['expected_grid_cells']
    if extent is None or expected_cells is None:
        raise RuntimeError(
            f"Extent for {city_config['city_name']} is not frozen. "
            'Run the standardized extent-discovery cell before any raster job.'
        )
    required = {'min_x', 'min_y', 'max_x', 'max_y'}
    if set(extent) != required:
        raise RuntimeError(f'Extent keys must be exactly {sorted(required)}')
    if not (extent['min_x'] < extent['max_x'] and extent['min_y'] < extent['max_y']):
        raise RuntimeError('Invalid extent ordering')
    if expected_cells <= 0:
        raise RuntimeError('expected_grid_cells must be positive')
    return True

print('Current city:', ACTIVE_CITY['city_name'])
print('Extent:', ACTIVE_CITY['extent_mollweide'])
print('Expected grid cells:', ACTIVE_CITY['expected_grid_cells'])
print('Raster processing remains BLOCKED until these are frozen.')


Current city: Hyderabad Metropolitan Region
Extent: None
Expected grid cells: None
Raster processing remains BLOCKED until these are frozen.


In [6]:
# CELL 6: STANDARDIZED HDFS PATH FACTORY

def city_paths(config, epoch=None):
    base = config['hdfs_output_base']
    result = {
        'base': base,
        'longitudinal': f'{base}/longitudinal_grid_year/all_epochs',
        'epoch_summary': f'{base}/historical_summaries/epoch_summary',
        'population_transitions': f'{base}/historical_comparisons/population_growth_transitions',
        'built_transitions': f'{base}/historical_comparisons/built_surface_transitions',
        'smod_transitions': f'{base}/historical_comparisons/smod_transitions',
        'smod_matrix': f'{base}/historical_summaries/smod_transition_matrix',
        'smod_full_matrix': f'{base}/historical_summaries/smod_1975_2020_matrix',
        'benchmarks': f'{base}/benchmarks',
    }
    if epoch is not None:
        result.update({
            'population': f'{base}/product=population/epoch={epoch}/grid_1km',
            'smod': f'{base}/product=smod/epoch={epoch}/grid_1km',
            'built_surface': f'{base}/product=built_surface/epoch={epoch}/grid_1km',
            'enriched': f'{base}/enriched_grid_year/epoch={epoch}',
        })
    return result

print(city_paths(ACTIVE_CITY, 1975))
print('CELL 6: SUCCESS')


{'base': 'hdfs://localhost:9000/data/ghsl/curated/cities/city=hyderabad', 'longitudinal': 'hdfs://localhost:9000/data/ghsl/curated/cities/city=hyderabad/longitudinal_grid_year/all_epochs', 'epoch_summary': 'hdfs://localhost:9000/data/ghsl/curated/cities/city=hyderabad/historical_summaries/epoch_summary', 'population_transitions': 'hdfs://localhost:9000/data/ghsl/curated/cities/city=hyderabad/historical_comparisons/population_growth_transitions', 'built_transitions': 'hdfs://localhost:9000/data/ghsl/curated/cities/city=hyderabad/historical_comparisons/built_surface_transitions', 'smod_transitions': 'hdfs://localhost:9000/data/ghsl/curated/cities/city=hyderabad/historical_comparisons/smod_transitions', 'smod_matrix': 'hdfs://localhost:9000/data/ghsl/curated/cities/city=hyderabad/historical_summaries/smod_transition_matrix', 'smod_full_matrix': 'hdfs://localhost:9000/data/ghsl/curated/cities/city=hyderabad/historical_summaries/smod_1975_2020_matrix', 'benchmarks': 'hdfs://localhost:9000/d

In [7]:
# CELL 7: EXECUTION PLAN AND CHECKPOINT POLICY

def build_execution_plan(config):
    plan = []
    for phase, epochs in [('proof', PROOF_EPOCHS), ('historical', REMAINING_EPOCHS)]:
        for epoch in epochs:
            paths = city_paths(config, epoch)
            for stage in ['population', 'smod', 'built_surface', 'enriched']:
                plan.append({
                    'city_slug': config['city_slug'],
                    'phase': phase,
                    'epoch': epoch,
                    'stage': stage,
                    'output_path': paths[stage],
                    'policy': 'validate_then_skip_else_compute',
                })
    return plan

execution_plan = build_execution_plan(ACTIVE_CITY)
print('Planned stages:', len(execution_plan))
print('Expected after complete city run: 40 validated outputs')
for item in execution_plan[:12]:
    print(item)
print('CELL 7: SUCCESS')


Planned stages: 40
Expected after complete city run: 40 validated outputs
{'city_slug': 'hyderabad', 'phase': 'proof', 'epoch': 1975, 'stage': 'population', 'output_path': 'hdfs://localhost:9000/data/ghsl/curated/cities/city=hyderabad/product=population/epoch=1975/grid_1km', 'policy': 'validate_then_skip_else_compute'}
{'city_slug': 'hyderabad', 'phase': 'proof', 'epoch': 1975, 'stage': 'smod', 'output_path': 'hdfs://localhost:9000/data/ghsl/curated/cities/city=hyderabad/product=smod/epoch=1975/grid_1km', 'policy': 'validate_then_skip_else_compute'}
{'city_slug': 'hyderabad', 'phase': 'proof', 'epoch': 1975, 'stage': 'built_surface', 'output_path': 'hdfs://localhost:9000/data/ghsl/curated/cities/city=hyderabad/product=built_surface/epoch=1975/grid_1km', 'policy': 'validate_then_skip_else_compute'}
{'city_slug': 'hyderabad', 'phase': 'proof', 'epoch': 1975, 'stage': 'enriched', 'output_path': 'hdfs://localhost:9000/data/ghsl/curated/cities/city=hyderabad/enriched_grid_year/epoch=1975', 

## Production-driver design

The stable logic from the Bengaluru notebooks must be consolidated into one remote driver with these factories:

- `create_population_program(config, epoch)`
- `create_smod_program(config, epoch)`
- `create_built_surface_program(config, epoch)`
- `create_enrichment_program(config, epoch)`
- `create_product_validator(config, epoch, product)`
- `create_enriched_validator(config, epoch)`
- `run_stage_with_checkpoint(config, epoch, stage)`
- `build_longitudinal(config)`
- `build_epoch_summary(config)`
- `build_population_transitions(config)`
- `build_built_transitions(config)`
- `build_smod_transitions(config)`
- `build_benchmark_manifest(config)`

The final driver must incorporate the successful Bengaluru fixes directly. It must not contain the patch cells from Notebook 04 or 05.

In [8]:
# CELL 8: CITY RUN GUARD
# This is intentionally the last executable cell in the initial common notebook.
# It prevents an accidental city batch before the extent and expected grid count
# are reviewed and frozen.

assert_extent_frozen(ACTIVE_CITY)

print('City:', ACTIVE_CITY['city_name'])
print('Extent:', ACTIVE_CITY['extent_mollweide'])
print('Expected grid cells:', ACTIVE_CITY['expected_grid_cells'])
print('Proof epochs authorized:', PROOF_EPOCHS)
print()
print('NEXT ACTION: generate and compile the common remote production driver.')
print('Do not launch remaining epochs until all four outputs pass for 1975, 2000, and 2020.')


RuntimeError: Extent for Hyderabad Metropolitan Region is not frozen. Run the standardized extent-discovery cell before any raster job.

# Completion checklist per city

A city is complete only when all conditions pass:

- 30 raw product/epoch combinations are available.
- 40 product and enriched HDFS outputs exist.
- Every output is readable as Parquet.
- Every output has the city-specific expected grid-cell count.
- Grid keys are non-null and unique.
- One longitudinal row exists for every `grid_cell_id × epoch`.
- Longitudinal rows equal `expected_grid_cells × 10`.
- Each transition dataset has `expected_grid_cells × 9` rows.
- Epoch summary has 10 rows.
- SMOD period and full-period matrices reconcile to expected grid cells.
- Benchmark manifest, dataset inventory, and runtime summary pass read-back validation.

## Training roles

- Training: Bengaluru, Hyderabad, Pune metro, Chennai metro, Mumbai metro, Delhi NCR, Ahmedabad, Surat.
- Held out: Jaipur.
- Ahmedabad and Surat are processed last as requested.